# FAF / IR / OTHER modality classifier: demo

Predicts whether a retinal image is **FAF**, **IR** or **OTHER**, using the DenseNet-121 model in `model/`.

Run this notebook from the `faf_ir_demo_share` folder (it imports `predict.py`). See `README.md` for setup.

## 1. Settings
Point `IMAGE_PATH` at a single image or a folder (folders are searched recursively).

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from predict import ModalityClassifier, find_images

CHECKPOINT_PATH = Path("model/final_model_pretrained.pt")
IMAGE_PATH = Path("/home/tdhar/image_faf_ir/images_test")   # change to your own image or folder
OUTPUT_CSV = Path("predictions.csv")

## 2. Load the model
Image size and circle-mask settings are read from the checkpoint, so preprocessing matches training.

In [ ]:
clf = ModalityClassifier(CHECKPOINT_PATH)
print(f"Model: {clf.model_name} | classes: {clf.classes} | image size: {clf.image_size} "
      f"| circle mask: {clf.circle_mask} | device: {clf.device}")

## 3. Predict

In [ ]:
files = find_images([IMAGE_PATH])
print(f"{len(files)} images found")

results = [clf.predict(f) for f in files]
predictions_df = pd.DataFrame([
    {"image": str(f), "prediction": r["predicted_label"], "confidence": round(r["confidence"], 4),
     **{f"p_{c}": round(p, 4) for c, p in r["probabilities"].items()}}
    for f, r in zip(files, results)
])
display(predictions_df)
print(predictions_df["prediction"].value_counts().to_string())
predictions_df.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV.resolve())

## 4. Show the images as the model sees them
Resized, padded and circle-masked, with the prediction and confidence. Shows the first `MAX_SHOWN` images.

In [ ]:
MAX_SHOWN = 24
columns = 4
shown = list(zip(files, results))[:MAX_SHOWN]
rows = max(1, -(-len(shown) // columns))
fig, axes = plt.subplots(rows, columns, figsize=(3.2 * columns, 3.4 * rows), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, (f, r) in zip(axes.flat, shown):
    ax.imshow(r["image"])
    ax.set_title(f"{f.name}\n{r['predicted_label']} ({r['confidence']:.1%})", fontsize=8)
plt.tight_layout()
plt.show()